<a href="https://colab.research.google.com/github/CyprianBohojlo/APO/blob/colab-port/colab_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APO Colab Bootstrap

This notebook sets up the environment for running ProTeGi-style prompt
optimization experiments on Colab Pro with data stored on Google Drive.

**Supported tasks:** FinanceBench, FinQA, FinDoc-RAG

**Prerequisites:**
- A **GPU runtime** (Runtime > Change runtime type > T4 GPU).
- An `OPENAI_API_KEY` stored in Colab Secrets (key icon in the left sidebar).
- Vectorstores uploaded to Google Drive (see the layout section below).
- Optionally, Kaggle credentials in Colab Secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`) if using the BEM scorer.

## 1. Mount Google Drive

In [21]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration

Edit `APO_ROOT` below to match where your project data lives on Drive.

The expected directory layout under `APO_ROOT` is:
```
data/
  FinanceBench/       # dataset_prepared.parquet + source jsonl files
  FinQa/              # dataset_prepared.parquet + train.json, dev.json, test.json
  Findoc/             # dataset_prepared.parquet + qa.json
references/
  FinanceBenchPdfs/   # downloaded automatically on first run
  FinDoc/             # markdown reference files
vectorstores/
  FinQa/finqa/<doc_id>/chroma.sqlite3   # one folder per document
  FinanceBench/financebench/<doc_id>/chroma.sqlite3
  FinDoc/findoc/<doc_id>/chroma.sqlite3
results/              # created automatically
prompts/              # seed prompt .txt files
```

In [22]:
import os

APO_ROOT = "/content/drive/MyDrive/Pubs/Articles/Automatic Prompt Optimization with RAG/Experiments"

os.environ["APO_ROOT"] = APO_ROOT
os.environ["HF_HOME"] = os.path.join(APO_ROOT, ".cache", "hf")

os.makedirs(APO_ROOT, exist_ok=True)
for subdir in ["data", "references", "vectorstores", "results", "prompts"]:
    os.makedirs(os.path.join(APO_ROOT, subdir), exist_ok=True)

print(f"APO_ROOT = {APO_ROOT}")
print(f"HF_HOME  = {os.environ['HF_HOME']}")

APO_ROOT = /content/drive/MyDrive/Pubs/Articles/Automatic Prompt Optimization with RAG/Experiments
HF_HOME  = /content/drive/MyDrive/Pubs/Articles/Automatic Prompt Optimization with RAG/Experiments/.cache/hf


## 3. Secrets

Store your `OPENAI_API_KEY` in **Colab Secrets** (the key icon in the
left sidebar). The cell below reads it from there so it never appears
in the notebook.

In [23]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY not found in Colab Secrets."
print("OPENAI_API_KEY loaded.")

# Kaggle credentials for the BEM scorer.
try:
    kaggle_user = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
    if kaggle_user and kaggle_key:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        import json
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
            json.dump({"username": kaggle_user, "key": kaggle_key}, f)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        print("Kaggle credentials written.")
    else:
        print("Kaggle credentials not set (BEM scorer will not be available).")
except Exception:
    print("Kaggle credentials not set (BEM scorer will not be available).")

OPENAI_API_KEY loaded.
Kaggle credentials written.


## 4. Clone the repo and install dependencies

In [24]:
REPO_DIR = "/content/APO"

if not os.path.isdir(REPO_DIR):
    !git clone -b colab-port https://github.com/CyprianBohojlo/APO.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 799 bytes | 399.00 KiB/s, done.
From https://github.com/CyprianBohojlo/APO
   6f48bf0..b222987  colab-port -> origin/colab-port
Updating 6f48bf0..b222987
Fast-forward
 colab_bootstrap.ipynb | 16 ++--------------
 generate.py           |  2 +-
 vectorize.py          |  4 ++--
 3 files changed, 5 insertions(+), 17 deletions(-)


## 5. Verify vectorstore layout

The pipeline expects per-document Chroma stores at:
```
{APO_ROOT}/vectorstores/{TaskDir}/{task}/{doc_id}/chroma.sqlite3
```
For FinQA this is `vectorstores/FinQa/finqa/<doc_id>/chroma.sqlite3`.

Run the cell below to check whether your vectorstores are in the
expected location. If the count is 0 but you have uploaded them
to a different Drive path, either move them or create a symlink.

In [25]:
import pathlib

TASK = "finqa"

task_vs_dirs = {
    "financebench": "FinanceBench/financebench",
    "finqa": "FinQa/finqa",
    "findoc": "FinDoc/findoc",
}

vs_path = pathlib.Path(APO_ROOT) / "vectorstores" / task_vs_dirs[TASK]
if vs_path.exists():
    docs = [d.name for d in vs_path.iterdir() if d.is_dir()]
    chroma_count = sum(1 for d in vs_path.iterdir() if (d / "chroma.sqlite3").exists())
    print(f"Found {len(docs)} doc folders, {chroma_count} with chroma.sqlite3")
    if docs:
        print(f"Sample: {docs[:5]}")
else:
    print(f"Directory does not exist: {vs_path}")
    print(f"Upload your vectorstores there, or symlink from wherever they live on Drive.")

Found 8281 doc folders, 8281 with chroma.sqlite3
Sample: ['AMT_2010_page_99.pdf-2_2b087e29fd64', 'AMT_2012_page_104.pdf-1_b105dceab804', 'AMT_2010_page_98.pdf-4_08d844831070', 'AMT_2010_page_98.pdf-3_12f76d74154b', 'AMT_2010_page_98.pdf-2_f92685717999']


## 6. Copy vectorstores to local disk

Google Drive I/O is slow. Copying vectorstores to the local Colab SSD
before running experiments gives a significant speedup on retrieval.
This takes a few minutes the first time but only needs to happen once
per session.

In [ ]:
import shutil, pathlib, time as _time

LOCAL_VS = "/content/vectorstores_local/FinQa"
src = pathlib.Path(APO_ROOT) / "vectorstores" / "FinQa"

if not pathlib.Path(LOCAL_VS).exists():
    print(f"Copying vectorstores from Drive to local disk...")
    t0 = _time.time()
    shutil.copytree(str(src), LOCAL_VS)
    print(f"Done in {(_time.time() - t0)/60:.1f} min.")
else:
    n = sum(1 for d in pathlib.Path(LOCAL_VS).rglob("chroma.sqlite3"))
    print(f"Local vectorstores already present ({n} stores).")

## 7. Run remaining FinQA experiments

This cell is **resumable**. It checks which experiments have already
completed (by counting ROUND entries in the output file) and skips
them. If the runtime crashes, just reconnect, re-run the setup cells,
and re-run this cell.

In [ ]:
import subprocess, sys, pathlib, time, threading

EXPECTED_ROUNDS = 7
MAX_THREADS = 8

experiments = [
    {"id": "E20", "evaluator": "ppo", "samples_per_eval": 24, "eval_rounds": 2, "eval_prompts_per_round": 2, "top_k": 1},
    {"id": "E22", "evaluator": "ucb", "samples_per_eval": 36, "eval_rounds": 4, "eval_prompts_per_round": 4, "top_k": 3},
    {"id": "E23", "evaluator": "ppo", "samples_per_eval": 36, "eval_rounds": 4, "eval_prompts_per_round": 4, "top_k": 3},
    {"id": "E24", "evaluator": "dpo", "samples_per_eval": 36, "eval_rounds": 4, "eval_prompts_per_round": 4, "top_k": 3},
    {"id": "E25", "evaluator": "ucb", "samples_per_eval": 48, "eval_rounds": 8, "eval_prompts_per_round": 8, "top_k": 6},
    {"id": "E26", "evaluator": "ppo", "samples_per_eval": 48, "eval_rounds": 8, "eval_prompts_per_round": 8, "top_k": 6},
    {"id": "E27", "evaluator": "dpo", "samples_per_eval": 48, "eval_rounds": 8, "eval_prompts_per_round": 8, "top_k": 6},
]

def count_rounds(filepath):
    p = pathlib.Path(filepath)
    if not p.exists():
        return 0
    return p.read_text().count("======== ROUND")

def tail_output(filepath, stop_event):
    p = pathlib.Path(filepath)
    last_rounds = 0
    while not stop_event.is_set():
        stop_event.wait(30)
        if not p.exists():
            continue
        try:
            text = p.read_text()
        except Exception:
            continue
        rounds = text.count("======== ROUND")
        if rounds != last_rounds:
            last_rounds = rounds
            lines = text.strip().split("\n")
            last_line = lines[-1] if lines else ""
            elapsed = time.time() - tail_output.start_time
            print(f"  [progress] Round {rounds}/{EXPECTED_ROUNDS} ({elapsed/60:.0f} min elapsed)  |  {last_line[:120]}", flush=True)

data_dir = f"{APO_ROOT}/data/FinQa"
prompt_file = f"{APO_ROOT}/prompts/basic.txt"

completed = 0
skipped = 0

for exp in experiments:
    out = f"{APO_ROOT}/results/{exp['id']}_finqa_{exp['evaluator']}.txt"
    rounds_done = count_rounds(out)
    if rounds_done >= EXPECTED_ROUNDS:
        print(f"SKIP {exp['id']} ({exp['evaluator']}, budget samples={exp['samples_per_eval']}): already complete ({rounds_done} rounds)")
        skipped += 1
        continue

    if rounds_done > 0:
        print(f"NOTE: {exp['id']} has {rounds_done}/{EXPECTED_ROUNDS} rounds but is incomplete. Restarting.")

    print(f"\n{'='*60}")
    print(f"STARTING {exp['id']}: finqa / {exp['evaluator']} / samples={exp['samples_per_eval']} / eval_rounds={exp['eval_rounds']} / threads={MAX_THREADS}")
    print(f"{'='*60}", flush=True)

    start = time.time()
    tail_output.start_time = start
    stop_event = threading.Event()
    monitor = threading.Thread(target=tail_output, args=(out, stop_event), daemon=True)
    monitor.start()

    result = subprocess.run(
        [
            sys.executable, "main.py",
            "--task", "finqa",
            "--data_dir", data_dir,
            "--prompts", prompt_file,
            "--evaluator", exp["evaluator"],
            "--samples_per_eval", str(exp["samples_per_eval"]),
            "--eval_rounds", str(exp["eval_rounds"]),
            "--eval_prompts_per_round", str(exp["eval_prompts_per_round"]),
            "--top_k", str(exp["top_k"]),
            "--max_threads", str(MAX_THREADS),
            "--out", out,
        ],
        env={**os.environ, "APO_ROOT": APO_ROOT},
        capture_output=True,
        text=True,
    )

    stop_event.set()
    monitor.join(timeout=5)
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f"\nFAILED {exp['id']} after {elapsed/60:.1f} min (exit code {result.returncode})")
        if result.stderr:
            stderr_lines = result.stderr.strip().split("\n")
            print("\n".join(stderr_lines[-15:]))
        print()
    else:
        completed += 1
        print(f"\nCOMPLETED {exp['id']} in {elapsed/60:.1f} min. Output: {out}")

print(f"\n{'='*60}")
print(f"Done. {completed} completed, {skipped} skipped, {len(experiments) - completed - skipped} failed.")
print(f"{'='*60}")